# Phase 2 - 03: GRPO 训练

## 本节目标

用 TRL 的 `GRPOTrainer` 对 Qwen3.5-2B 做强化学习微调，让模型学会：
1. 稳定输出 `<think>` / `<tool_call>` 两段式格式
2. 在 `<tool_call>` 里写真实可执行的 Python 表达式
3. `<tool_call>` 的执行结果与 GSM8K 正确答案一致

**设计思路**：答案由工具（calculator）给出，不要求模型自己算——这才是 tool-use 的正确训练方式。
模型只需要负责"把题目翻译成正确的 Python 表达式"，验证答案的工作交给执行器。

## GRPO 简介

GRPO（Group Relative Policy Optimization）是 DeepSeek-R1 用的算法，比 PPO 简单：
- 同一道题生成 G 个回答（一个「组」）
- 组内做相对比较，reward 高于均值的回答得到正梯度，低于均值的得到负梯度
- 不需要单独训练 Critic 网络，省显存

```
advantage_i = (reward_i - mean(rewards)) / std(rewards)
```

**std（标准差）的计算方式：**

```
std = sqrt( mean( (x_i - mean(x))² ) )
```

以一组 reward `[0.1, 0.3, 1.5, 0.2]` 为例：

```
mean     = (0.1 + 0.3 + 1.5 + 0.2) / 4 = 0.525

各项偏差²:
  (0.1 - 0.525)² = 0.1806
  (0.3 - 0.525)² = 0.0506
  (1.5 - 0.525)² = 0.9506
  (0.2 - 0.525)² = 0.1056

variance = mean([0.1806, 0.0506, 0.9506, 0.1056]) = 0.3219
std      = sqrt(0.3219) ≈ 0.567

advantages = [(-0.74), (-0.40), (+1.72), (-0.57)]
```

直觉：std 衡量「这组回答的差异有多大」。如果 4 个回答 reward 完全相同，std=0，没有训练信号（实现上会加一个小 epsilon 防止除零）。

## 1. 导入依赖 & 复用工具函数（从 01/02 搬来）

In [1]:
# 1-1 导入依赖 & 工具函数
import re
import math
import json
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
# https://hugging-face.cn/docs/trl/index
# TRL - Transformer Reinforcement Learning
# TRL 是一个完整的堆栈库，提供了一套工具来使用诸如监督微调 (SFT)、群组相对策略优化 (GRPO)、
# 直接偏好优化 (DPO)、奖励建模等方法来训练 transformer 语言模型。
from datasets import Dataset

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
MODEL_ID = os.path.expanduser("~/.cache/modelscope/hub/models/Qwen/Qwen3___5-2B")

print(f"torch:   {torch.__version__}")
print(f"device:  {DEVICE}")

# ── 计算器 ────────────────────────────────────────────────────────────────────
SAFE_MATH_ENV = {
    "__builtins__": {},
    "abs": abs, "round": round, "int": int, "float": float,
    "sqrt": math.sqrt, "pow": math.pow, "floor": math.floor, "ceil": math.ceil,
}

def calculator(expression: str) -> str:
    try:
        result = eval(expression.strip().replace("^", "**"), SAFE_MATH_ENV)
        if isinstance(result, float) and result.is_integer():
            return str(int(result))
        return str(result)
    except Exception as e:
        return f"ERROR: {e}"

# ── 解析输出 ──────────────────────────────────────────────────────────────────
def parse_output(text: str) -> dict:
    result = {"think": None, "tool_call": None, "answer": None}
    for tag in ["think", "tool_call", "answer"]:
        m = re.search(rf"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
        if m:
            result[tag] = m.group(1).strip()
    return result

# ── 答案归一化 ────────────────────────────────────────────────────────────────
def normalize_answer(s: str) -> str:
    if s is None:
        return ""
    s = re.sub(r"[,，]", "", s.strip())
    s = re.sub(r"[^0-9\.\-]", "", s)
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except:
        return s

print("工具函数加载完成")

/opt/homebrew/Caskroom/miniconda/base/envs/rl_study/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch:   2.10.0
device:  mps
工具函数加载完成


## 2. Reward 函数

Reward 分三层：

| 层级 | 条件 | 得分 |
|---|---|---|
| 格式分 | `<think>` 存在 | +0.1 |
| 格式分 | `<tool_call>` 存在 | +0.1 |
| 工具分 | `tool_call` 内容可被 `eval()` 执行 | +0.2 |
| 结果分 | `calculator(tool_call)` == GSM8K 正确答案 | +1.0 |

**设计要点**：结果分直接验证 `calculator(tool_call)` 的执行结果，而不是要求模型自己在 `<answer>` 里写出答案。
模型只需要写出正确的 Python 表达式，"算出来"的工作完全交给执行器。

In [2]:
# 2-1 Reward 函数
def format_reward(output: str) -> float:
    """两个标签各 0.1，满分 0.2"""
    parsed = parse_output(output)
    score = 0.0
    if parsed["think"] is not None:     score += 0.1
    if parsed["tool_call"] is not None: score += 0.1
    return round(score, 2)


def tool_execution_reward(output: str) -> float:
    """tool_call 可执行得 0.2，否则 0"""
    parsed = parse_output(output)
    if not parsed["tool_call"]:
        return 0.0
    return 0.2 if not calculator(parsed["tool_call"]).startswith("ERROR") else 0.0


def tool_result_reward(output: str, correct_answer: str) -> float:
    """
    calculator(tool_call) 执行结果与 GSM8K 正确答案一致得 1.0，否则 0。
    这是 tool-use 的正确验证方式：模型只需写出正确的 Python 表达式，
    答案由执行器（calculator）给出，不要求模型自己计算。
    """
    parsed = parse_output(output)
    if not parsed["tool_call"]:
        return 0.0
    result = calculator(parsed["tool_call"])
    if result.startswith("ERROR"):
        return 0.0
    return 1.0 if normalize_answer(result) == normalize_answer(correct_answer) else 0.0


def compute_reward(output: str, correct_answer: str) -> dict:
    r_format = format_reward(output)
    r_tool   = tool_execution_reward(output)
    r_result = tool_result_reward(output, correct_answer)
    return {
        "format": r_format,
        "tool":   r_tool,
        "result": r_result,
        "total":  round(r_format + r_tool + r_result, 2),
    }


def _completion_to_str(completion) -> str:
    """新版 TRL 的 completion 可能是 message list，提取最后一条 assistant 消息的文本"""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        for msg in reversed(completion):
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                return msg.get("content", "")
        return " ".join(msg.get("content", "") for msg in completion if isinstance(msg, dict))
    return str(completion)


# GRPOTrainer 接口：completions 列表 → rewards 列表
def reward_fn(completions: list, **kwargs) -> list[float]:
    answers = kwargs["answer"]
    texts = [_completion_to_str(c) for c in completions]
    return [compute_reward(t, a)["total"] for t, a in zip(texts, answers)]


# 快速验证
r = compute_reward(
    "<think>先算 48+24=72</think><tool_call>48 + 48/2</tool_call>",
    "72"
)
print("完美回答 reward:", r)

r2 = compute_reward(
    "<think>分析</think><tool_call>16 - 3 - 4</tool_call>",
    "18"
)
print("表达式错误 reward:", r2)

r3 = compute_reward("16 - 3 - 4 = 9\n9 * 2 = 18\n18", "18")
print("无格式直接答对:", r3)

完美回答 reward: {'format': 0.2, 'tool': 0.2, 'result': 1.0, 'total': 1.4}
表达式错误 reward: {'format': 0.2, 'tool': 0.2, 'result': 0.0, 'total': 0.4}
无格式直接答对: {'format': 0.0, 'tool': 0.0, 'result': 0.0, 'total': 0.0}


## 3. 准备数据集

GRPOTrainer 需要一个 HuggingFace `Dataset`，每条记录包含：
- `prompt`：发给模型的消息列表（chat 格式）
- `answer`：正确答案（字符串），reward_fn 通过 `**kwargs` 拿到

In [3]:
# 3-1 准备数据集
SYSTEM_PROMPT = """你是一个数学助手，必须调用计算器工具解决计算问题。

严格按照以下格式输出，不得添加任何额外文字：

<think>
分析题目，写出计算思路
</think>
<tool_call>
只写一个可以被 Python eval() 直接执行的数学表达式，不写任何中文说明
</tool_call>

重要规则：
- tool_call 标签内只能有数字、运算符（+ - * / ** ()）和空格
- 绝对不能在 tool_call 里写中文或英文说明文字
- 输出 tool_call 后停止，不要写其他内容

示例：
问题：小明有 12 个苹果，吃了 4 个，又买了 7 个，还剩多少？
<think>
先用 12 减去 4，再加 7。
</think>
<tool_call>
12 - 4 + 7
</tool_call>
"""

GSM8K_DIR = os.path.expanduser("~/.cache/modelscope/datasets/gsm8k/data")

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def extract_gsm8k_answer(answer_str: str) -> str:
    m = re.search(r"####\s*([\d,]+)", answer_str)
    return m.group(1).replace(",", "") if m else None

def make_dataset(raw_data: list, max_samples: int = None) -> Dataset:
    """把 GSM8K 原始数据转换为 GRPOTrainer 所需格式"""
    if max_samples:
        raw_data = raw_data[:max_samples]
    records = []
    for item in raw_data:
        answer = extract_gsm8k_answer(item["answer"])
        if answer is None:
            continue
        records.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": item["question"]},
            ],
            "answer": answer,
        })
    return Dataset.from_list(records)


gsm8k_train_raw = load_jsonl(f"{GSM8K_DIR}/train.jsonl")
gsm8k_test_raw  = load_jsonl(f"{GSM8K_DIR}/test.jsonl")

# 训练用前 2000 条（200 → 2000，增大数据多样性）
train_dataset = make_dataset(gsm8k_train_raw, max_samples=2000)
eval_dataset  = make_dataset(gsm8k_test_raw,  max_samples=50)

print(f"训练集: {len(train_dataset)} 条")
print(f"验证集: {len(eval_dataset)} 条")
print("\n样本示例:")
print("  answer:", train_dataset[0]["answer"])
print("  prompt[1]:", train_dataset[0]["prompt"][1]["content"][:60], "...")

训练集: 2000 条
验证集: 50 条

样本示例:
  answer: 72
  prompt[1]: Natalia sold clips to 48 of her friends in April, and then s ...


## 4a. 加载模型

In [4]:
# 4-1 加载模型
print("加载 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("tokenizer 加载完成")

print("加载模型...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map=DEVICE,
    local_files_only=True,
)
model.gradient_checkpointing_enable()  # 省激活值显存

param_count = sum(p.numel() for p in model.parameters()) / 1e9
print(f"模型加载完成，参数量: {param_count:.2f}B，设备: {next(model.parameters()).device}")

加载 tokenizer...


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


tokenizer 加载完成
加载模型...


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 320/320 [00:03<00:00, 80.19it/s]

模型加载完成，参数量: 1.88B，设备: mps:0


## 4b. 用 LoRA 包装模型（解决 OOM）

全量训练 1.88B 模型时，梯度 + 优化器状态需要约 2x 模型显存，M4 16G 装不下。

LoRA 的做法：冻结原始权重，只在 attention 的 Q/V 矩阵旁插入两个小矩阵（rank=8），
可训练参数量从 1.88B 降到约 **10M**（0.5%），显存需求大幅下降。

```
W_new = W_frozen + BA    (B: d×r, A: r×d, r=8 << d=2048)
```

In [5]:
# 4-2 LoRA 包装模型
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                                    # LoRA rank，越小越省显存，越大效果越好
    lora_alpha=16,                          # 缩放系数，通常设为 2*r
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],    # 只在 Q/V 矩阵插入适配器
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 417,792 || all params: 1,882,242,880 || trainable%: 0.0222


## 5a. 配置 GRPOTrainer

关键参数说明：

| 参数 | 值 | 说明 |
|---|---|---|
| `num_generations` | 4 | 每道题生成 4 个回答（组大小 G=4） |
| `max_completion_length` | 256 | 每次生成最多 256 个 token（原名 `max_new_tokens`，TRL 0.29 改名）。限制这个是为了防止模型无限生成，同时控制训练速度——GRPO 每步要生成 4 个回答，上限越大单步越慢 |
| `learning_rate` | 5e-6 | 较小的学习率，避免训练初期震荡 |
| `per_device_train_batch_size` | 1 | 显存限制决定的，不是算出来的。MPS 上模型权重 + 梯度 + 激活值已接近满载，设为 1 是当前上限，改成 2 会 OOM |
| `gradient_accumulation_steps` | 4 | 连续跑 4 个 batch=1 再更新一次参数，等效 batch=4，但显存峰值不变 |
| `max_steps` | 200 | 参数更新次数上限，每步取 1 条数据。epoch 没有直接控制，通过 max_steps 间接体现：200步 = 200/2000 = 0.1 个 epoch；跑满 1 epoch 需要 max_steps=2000 |

In [8]:
# 5-1 配置 GRPOTrainer 参数
training_args = GRPOConfig(
    output_dir="./checkpoints/grpo_qwen35_gsm8k",

    # GRPO 核心参数
    num_generations=4,           # 每题生成 4 个回答做组内对比
    max_completion_length=256,   # 256 省显存，三段式输出够用
    temperature=0.8,             # 生成时加一点随机性，让组内有差异

    # 训练超参
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=200,               # 先跑 200 步验证趋势，确认上升后再改到 800
    warmup_steps=20,
    optim="adafactor",           # 不存完整动量，省约 2x 模型显存（Adam OOM 的根本原因）

    # 日志 & 保存
    logging_steps=10,
    save_steps=200,
    report_to="none",            # 不用 wandb

    # 其他
    remove_unused_columns=False,  # 保留 answer 列供 reward_fn 使用
    seed=42,
)

print("GRPOConfig 配置完成")
print(f"训练步数: {training_args.max_steps}")
print(f"每步生成: {training_args.num_generations} 个回答")

GRPOConfig 配置完成
训练步数: 200
每步生成: 4 个回答


## 5b. 推理工具函数（baseline & 训练后评估复用）

In [11]:
# 5-2 推理工具函数
def generate(question: str, max_new_tokens: int = 512) -> str:
    """
    用当前模型推理（baseline 和训练后评估都调用这个函数）。
    max_new_tokens 与训练时的 max_completion_length 作用相同（都限制生成长度），
    但这里用 512 而非 256——评估时不需要省显存，允许模型输出更完整的内容。
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,    # 评估时用低温，减少随机性
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            # use_cache=False 只在训练时需要（兼容 gradient checkpointing），eval 不需要
        )
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


def run_eval(label: str, samples: list) -> list[dict]:
    """对 samples 列表跑评估，打印结果，返回每道题的 reward 详情。"""
    import time
    print(f"=== {label} ===\n")
    results = []
    for i, (question, correct) in enumerate(samples):
        t0 = time.time()
        output = generate(question)
        elapsed = time.time() - t0
        r = compute_reward(output, correct)
        parsed = parse_output(output)
        tool_result = calculator(parsed["tool_call"]) if parsed["tool_call"] else "N/A"
        print(f"[题目 {i+1}] 正确答案: {correct}  耗时: {elapsed:.1f}s")
        print(f"  think:      {parsed['think']}")
        print(f"  tool_call:  {parsed['tool_call']}")
        print(f"  计算器结果: {tool_result}")
        print(f"  reward:  format={r['format']}  tool={r['tool']}  result={r['result']}  total={r['total']}")
        print()
        results.append(r)
    avg = sum(r["total"] for r in results) / len(results)
    print(f"平均 reward: {avg:.3f}\n")
    return results

print("generate / run_eval 函数定义完成")

generate / run_eval 函数定义完成


## 6. Baseline 评估（训练前）

训练前先跑一次评估，记录未训练模型的 reward，作为对比基准。

In [12]:
# 6-1 Baseline 评估（训练前）
eval_samples = [(gsm8k_test_raw[i]["question"], extract_gsm8k_answer(gsm8k_test_raw[i]["answer"]))
                for i in range(3)]

baseline_results = run_eval("Baseline（未训练）", eval_samples)

=== Baseline（未训练） ===

[题目 1] 正确答案: 18  耗时: 59.2s
  think:      
  tool_call:  16 - 3 - 4
  计算器结果: 9
  reward:  format=0.2  tool=0.2  result=0.0  total=0.4

[题目 2] 正确答案: 3  耗时: 76.2s
  think:      
  tool_call:  2 + 1
  计算器结果: 3
  reward:  format=0.2  tool=0.2  result=1.0  total=1.4

[题目 3] 正确答案: 70000  耗时: 9.1s
  think:      
  tool_call:  80000 + 50000 * 1.5 - 80000
  计算器结果: 75000
  reward:  format=0.2  tool=0.2  result=0.0  total=0.4

平均 reward: 0.733



## 7. 开始训练

MPS 上 200 步预计需要 **2-3 小时**（约 50 秒/步）。

训练日志里有 `rewards/mean` 字段，但 Jupyter 进度条默认只显示 `loss`。
下面用自定义 callback 在每个 logging step 额外打印一行 reward，方便观察训练是否有效。

In [ ]:
# 7-1 配置 callback + 初始化 trainer
import time
from transformers import TrainerCallback

class RewardLogCallback(TrainerCallback):
    """每个 logging step 打印 reward，补充进度条里看不到的信息。"""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        # TRL 0.29 的 key 是 rewards/reward_fn/mean，兼容旧版 rewards/mean
        r_mean = logs.get("rewards/reward_fn/mean") or logs.get("rewards/mean") or logs.get("reward")
        if r_mean is None:
            return
        r_std  = logs.get("rewards/reward_fn/std") or logs.get("rewards/std", 0)
        loss   = logs.get("loss", float("nan"))
        step   = state.global_step
        print(f"  step {step:4d} | loss={loss:.4f} | reward_mean={r_mean:.4f} | reward_std={r_std:.4f}")

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    callbacks=[RewardLogCallback()],
)

print("开始训练...")
t0 = time.time()

train_result = trainer.train()

elapsed = time.time() - t0
print(f"\n训练完成！耗时 {elapsed/60:.1f} 分钟")
print("训练指标:", train_result.metrics)

## 8. 训练后评估

In [ ]:
# 8-1 训练后评估
trained_results = run_eval("训练后", eval_samples)

# 对比 baseline vs 训练后
print("=== 对比 ===")
print(f"{'题目':<6} {'baseline':>10} {'训练后':>10} {'变化':>10}")
for i, (b, t) in enumerate(zip(baseline_results, trained_results)):
    delta = t["total"] - b["total"]
    print(f"题目 {i+1}   {b['total']:>10.2f} {t['total']:>10.2f} {delta:>+10.2f}")
avg_b = sum(r["total"] for r in baseline_results) / len(baseline_results)
avg_t = sum(r["total"] for r in trained_results) / len(trained_results)
print(f"{'平均':<6} {avg_b:>10.3f} {avg_t:>10.3f} {avg_t - avg_b:>+10.3f}")

## 9. Reward 曲线可视化

从训练日志中提取 reward 均值，画出训练过程中的变化趋势。

In [ ]:
# 9-1 Reward 曲线可视化
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Arial Unicode MS'

log_history = trainer.state.log_history

steps, rewards = [], []
for entry in log_history:
    # TRL 0.29 key 是 rewards/reward_fn/mean，兼容旧版
    reward_val = (entry.get("rewards/reward_fn/mean")
                  or entry.get("rewards/mean")
                  or entry.get("reward"))
    if reward_val is not None and "step" in entry:
        steps.append(entry["step"])
        rewards.append(reward_val)

if steps:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(steps, rewards, marker="o", markersize=4, linewidth=1.5, color="#2ecc71")
    ax.axhline(y=avg_b, color="#e74c3c", linestyle="--", alpha=0.6, label=f"baseline ({avg_b:.3f})")
    ax.set_xlabel("训练步数")
    ax.set_ylabel("平均 Reward")
    ax.set_title("GRPO 训练过程 — Reward 变化")
    ax.legend()
    ax.set_ylim(0, 1.6)
    plt.tight_layout()
    plt.show()
    print(f"起始 reward: {rewards[0]:.3f}")
    print(f"最终 reward: {rewards[-1]:.3f}")
    print(f"提升幅度:    {rewards[-1] - rewards[0]:+.3f}")
else:
    print("日志中未找到 reward 数据，log_history 样本：")
    print(log_history[:2])

## 10. 保存模型

In [ ]:
# 10-1 保存模型
SAVE_PATH = "./checkpoints/grpo_qwen35_gsm8k_final"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"模型已保存到 {SAVE_PATH}")

## 11. 小结

| 阶段 | 目标 | 状态 |
|---|---|---|
| 01 环境验证 | 模型加载、基础推理、baseline 测试 | 完成 |
| 02 Reward 设计 | 格式/工具/结果三层奖励 | 完成 |
| 03 GRPO 训练 | 训练循环、reward 提升 | 完成 |

---

## 训练实验记录

### 实验一：200条数据 / 300步 / V1 Prompt（三标签）
- 起始 reward: 0.040 → 最终 reward: 0.265，提升 +0.225
- 模型学会了输出标签格式，但 `tool_call` 里仍是 placeholder 文字，未学会写真实 Python 表达式
- **根因**：V1 System Prompt 里的示例文字（`Python 数学表达式（例如：15 - 7 + 3）`）被模型当模板直接复制，`tool_execution_reward` 永远为 0

### 实验二：2000条数据 / 200步 / V2 Prompt（三标签，含 answer）
- 起始 reward: 0.34，step 10 已达 0.58
- `tool_call` 全部变成了真实 Python 表达式（V2 Prompt 修复有效）
- 但答案正确率仍然低——表达式写出来了，但推理本身是错的
- **根因**：要求模型同时写 `<tool_call>` 和 `<answer>`，逻辑矛盾：tool-use 的答案应由工具给出，不应由模型自己算

### 实验三：2000条数据 / V3 Prompt（两标签，calculator 验证）
- 删除 `<answer>` 标签，reward 改为验证 `calculator(tool_call)` vs GSM8K 答案
- 新增 baseline 评估（训练前/训练后对比），量化训练效果
- Baseline 结果（3道题）：think 全为空，平均 reward 0.733（其中1题未训练就答对）

---

## 方案根本性问题分析

**当前方案（GSM8K + 单次 tool_call）不可行，原因：**

1. **think 为空，推理没有发生**：baseline 测试中 3 道题 think 全为空，模型没有在思考，只是在猜表达式

2. **训练信号稀疏**：
   - 简单题：未训练也能答对 → 组内4个回答reward相同，std=0，无训练信号
   - 难题：怎么训练也写不对 → reward 一直为0，也无训练信号
   - 有效信号只来自"偶尔能对、偶尔不对"的中间难度题，占比极少

3. **任务设计与 tool-use 目标不符**：
   - GSM8K 需要多步推理才能写出正确表达式，模型负担没有被工具减轻
   - 真实 tool-use 应该是"工具做模型做不到的事"（如大数计算），而不是"工具替模型做推理"
   - 强制每题都调用工具，也不符合真实场景（应该是模型自主判断是否需要工具）

**下一步方向**：重新设计任务，使用大数计算题（如 `1847 * 293`），工具真正减轻模型负担，训练信号密集。

---

## 踩坑记录

### 1. MPS OOM——优化器状态（Adam）
- **问题**：`optimizer.step()` 时 OOM
- **原因**：Adam 需要存两份动量，约等于 2x 模型参数量的额外显存
- **修复**：`optim="adafactor"`

### 2. MPS OOM——全量训练梯度
- **问题**：换 Adafactor 后仍 OOM
- **原因**：全量反向传播梯度本身需要 ~3.7 GiB
- **修复**：引入 LoRA（r=8，target Q/V），可训练参数降至 417K，加 `gradient_checkpointing_enable()`

### 3. 模型绕过工具直接答题（Reward Hacking）
- **修复**：无 `tool_call` 标签则结果分为 0

### 4. System Prompt Placeholder 被模型当模板复制
- **修复**：V2 Prompt 改为明确规则 + 独立 few-shot 示例

### 5. RewardLogCallback key 名不匹配
- **修复**：callback 里兼容 `rewards/reward_fn/mean` 和 `rewards/mean` 两种 key

### 6. Reward 设计逻辑矛盾（三标签 vs tool-use）
- **修复**：删除 `<answer>` 标签，改为验证 `calculator(tool_call)` 执行结果

### 7. eval 极慢（400s/题）
- **原因**：`use_cache=False` 禁用了 KV cache，该参数只为兼容训练时的 gradient checkpointing，eval 不需要
- **修复**：eval 的 `generate()` 中去掉 `use_cache=False`

---

## 下一步方向

- **重新设计任务**：用大数计算题替代 GSM8K，让工具真正减轻模型负担
- **选择性 tool-use**：混入不需要工具的题目，让模型学会"何时调用工具"
- **提高 LoRA rank**：r=8 → r=32
- **SFT 预训练**：先教会格式，再用 GRPO 强化